# EECE 5554 System Design Exercise: S9 Wearable Navigation Aid

**Zhiqi Zhang.** Scenario S9: a body-worn system guides a blind pedestrian from a bus stop to a room inside a building, with vibration cues, and warns of obstacles from the ground to head height.

| ID | Given requirement |
|---|---|
| G1 | Warn of any obstacle from the ground to head height within 3 m, at a walking speed of 1.4 m/s |
| G2 | Arrive within 1 m of the room door |
| G3 | All worn parts together weigh under 300 g and run 12 h on one charge |
| G4 | The user never needs to look at or hold a phone |

Each Part below runs the course tools with the numbers of this design. Sources for every input are in the design journal and in the report.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

K95 = {'1D': 1.96, '2D': 2.45}

FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

# Part 1: Problem Definition

## Tool 1: reaction time and warning range

A person has no fixed braking deceleration, so the human part of the stop is taken from measured unplanned stops (E8), counted from the cue, which already includes the reaction:

$$d_{detect} \ge v\,t_{system} + d_{human} + d_{margin}$$

For an obstacle moving toward the user at $v_o$ that does not yield, it keeps moving through the system latency and the user's stop time $t_{human}$:

$$d_{detect} \ge (v + v_o)\,t_{system} + d_{human} + v_o\,t_{human} + d_{margin}$$

In [ ]:
def latency_budget(stages):
    """Total latency in s from a dict of stage name to seconds."""
    return sum(stages.values()), stages


def warning_range(v, t_system, d_human, t_human, margin, v_obstacle=0.0):
    """Range in m at which a warning must start so that the user stops margin m short."""
    return (v + v_obstacle) * t_system + d_human + v_obstacle * t_human + margin

In [ ]:
V_USER = 1.4                         # m/s, G1
D_HUMAN = 1.113 * (1.4 / 1.22) ** 2  # m: E8 control group, mean + 2 SD = 0.669 + 2 x 0.222, scaled from 1.22 m/s by speed squared
T_HUMAN = 1.17 + 2 * 0.41            # s: E8 control group stopping time, mean + 2 SD
MARGIN = 0.15                        # m
V_PEDESTRIAN = 1.31                  # m/s, E9 usual outdoor walking pace
V_VEHICLE = 20 * 0.44704             # m/s, 20 mph, E11 upper bound for vehicles entering a driveway

t_near, near = latency_budget({'ToF frame, 15 Hz': 1 / 15, 'detection on the safety MCU': 0.02,
                               'BLE to the wristband': 0.05, 'motor start': 0.05})
t_far, far = latency_budget({'radar frame, 10 Hz': 0.10, 'tracking on the phone': 0.10,
                             'BLE chest to phone to wristband': 0.10, 'motor start': 0.05})
for name, stages, total in (('Near field', near, t_near), ('Far field', far, t_far)):
    print(name)
    for stage, t in stages.items():
        print(f'   {stage:32s} {1000 * t:5.0f} ms')
    print(f'   {"total":32s} {1000 * total:5.0f} ms')

print(f'Human stop from the cue: {D_HUMAN:.3f} m in {T_HUMAN:.2f} s')
static = warning_range(V_USER, t_near, D_HUMAN, T_HUMAN, MARGIN)
person = warning_range(V_USER, t_far, D_HUMAN, T_HUMAN, MARGIN, V_PEDESTRIAN)
vehicle = warning_range(V_USER, t_far, D_HUMAN, T_HUMAN, MARGIN, V_VEHICLE)
print(f'Static obstacle: {static:.2f} m needed, G1 gives 3 m')
print(f'R6: latency limit for a static obstacle at 3 m: {(3 - D_HUMAN - MARGIN) / V_USER:.3f} s')
print(f'R7: oncoming person at {V_PEDESTRIAN} m/s: {person:.2f} m')
print(f'R8: vehicle at {V_VEHICLE:.2f} m/s: {vehicle:.2f} m')

ts = np.linspace(0, 1.2, 121)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts, warning_range(V_USER, ts, D_HUMAN, T_HUMAN, MARGIN), label='static obstacle (R1, R6)')
ax.plot(ts, warning_range(V_USER, ts, D_HUMAN, T_HUMAN, MARGIN, V_PEDESTRIAN), label='oncoming pedestrian, 1.31 m/s (R7)')
ax.plot(ts, warning_range(V_USER, ts, D_HUMAN, T_HUMAN, MARGIN, V_VEHICLE), label='vehicle, 20 mph (R8)')
ax.axhline(3, color='gray', ls='--', lw=1, label='G1: 3 m')
ax.axhline(4, color='tab:purple', ls=':', lw=1, label='ToF maximum range, 4 m')
ax.axvline(t_near, color='k', ls=':', lw=0.8)
ax.axvline(t_far, color='k', ls='-.', lw=0.8)
ax.set_yscale('log')
ax.set_xlabel('system latency t_system (s)')
ax.set_ylabel('required warning range (m)')
ax.set_title('Warning range needed, user at 1.4 m/s')
ax.grid(alpha=0.3, which='both')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / 'tool1_warning_range.png', dpi=200)
plt.show()

# Part 3: Sensor Selection

## Tool 2: coverage and blind zones

Two VL53L5CX ToF sensors on the chest unit at 1.25 m: 45° × 45° field of view, 8 × 8 zones, 4 m range (ST datasheet DS13754). The search finds the pair of pitches that covers the ground to head height (1.65 m) from 1 to 3 m with the smallest ground blind zone.

In [ ]:
def vertical_window(height, pitch_down_deg, vfov_deg, d):
    """Lowest and highest heights in m that a sensor sees at horizontal distance d m."""
    lower = np.radians(pitch_down_deg + vfov_deg / 2.0)
    upper = np.radians(pitch_down_deg - vfov_deg / 2.0)
    d = np.asarray(d, dtype=float)
    return np.maximum(height - d * np.tan(lower), 0.0), height - d * np.tan(upper)


def ground_blind_zone(height, pitch_down_deg, vfov_deg):
    """Distance in m from the sensor to the first point on the ground it can see."""
    lower = np.radians(pitch_down_deg + vfov_deg / 2.0)
    return height / np.tan(lower) if lower > 0 else np.inf


def hfov_needed(width, d):
    """Horizontal field of view in degrees that covers a path width m wide at distance d m."""
    return np.degrees(2.0 * np.arctan(width / (2.0 * d)))


def points_on_target(target_width, d, angular_res_deg):
    """Number of scan points or zones that land on a target target_width m wide at d m."""
    return target_width / (d * np.radians(angular_res_deg))

In [ ]:
H_CHEST, VFOV, HEAD = 1.25, 45.0, 1.65
best = None
for p_low in np.arange(20, 70, 0.5):
    for p_high in np.arange(-30, 20, 0.5):
        ok = True
        for d in np.linspace(1.0, 3.0, 41):
            l1, u1 = vertical_window(H_CHEST, p_low, VFOV, d)
            l2, u2 = vertical_window(H_CHEST, p_high, VFOV, d)
            if not (l1 <= 0.0 and u1 >= l2 and u2 >= HEAD):
                ok = False
                break
        if ok:
            bz = ground_blind_zone(H_CHEST, p_low, VFOV)
            if best is None or bz < best[2]:
                best = (p_low, p_high, bz)
P_LOW, P_HIGH, BLIND = best
head_from = next(d for d in np.linspace(0.2, 3, 281) if vertical_window(H_CHEST, P_HIGH, VFOV, d)[1] >= HEAD)
print(f'Low sensor {P_LOW:.1f} deg down, high sensor {P_HIGH:.1f} deg down')
print(f'Ground first visible at {BLIND:.2f} m; head height first visible at {head_from:.2f} m')
print(f'HFOV needed for a 0.6 m path: {hfov_needed(0.6, 3.0):.1f} deg at 3 m, {hfov_needed(0.6, 1.0):.1f} deg at 1 m; the sensor has 45 deg')
print(f'Zones on a 5 cm pole at 3 m: {points_on_target(0.05, 3.0, 45 / 8):.2f}; on a 0.45 m wide person: {points_on_target(0.45, 3.0, 45 / 8):.2f}')
ray = np.arctan(H_CHEST / 3.0)
for drop in (0.013, 0.150):
    print(f'A {1000 * drop:.0f} mm drop at 3 m changes the ground range by {100 * drop / np.sin(ray):.1f} cm')

d = np.linspace(0, 4, 401)
l1, u1 = vertical_window(H_CHEST, P_LOW, VFOV, d)
l2, u2 = vertical_window(H_CHEST, P_HIGH, VFOV, d)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.fill_between(d, l1, u1, alpha=0.3, label=f'ToF low, {P_LOW:.1f} deg down')
ax.fill_between(d, l2, u2, alpha=0.3, label=f'ToF high, {P_HIGH:.1f} deg down')
ax.axhline(HEAD, color='tab:red', ls='--', lw=1, label='head height 1.65 m')
ax.axvline(3, color='gray', ls='--', lw=1)
ax.plot(0, H_CHEST, 'ks', label='chest unit, 1.25 m')
ax.set_xlim(0, 4)
ax.set_ylim(0, 2.2)
ax.set_xlabel('distance ahead (m)')
ax.set_ylabel('height (m)')
ax.set_title('Side view: two ToF sensors cover ground to head from 1 m to 3 m')
ax.legend(fontsize=7, loc='upper left')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / 'tool2_coverage.png', dpi=200)
plt.show()

In [ ]:
TOF_HFOV, TOF_RANGE, PATH_W = 45.0, 4.0, 0.6
full_path = (PATH_W / 2) / np.tan(np.radians(TOF_HFOV / 2))
print(f'Both ToF sensors face straight ahead; each covers {TOF_HFOV:.0f} deg horizontally out to {TOF_RANGE:.0f} m')
print(f'The full {PATH_W} m path is inside the field of view from {full_path:.2f} m ahead')
print(f'Field-of-view width: {2 * 1.0 * np.tan(np.radians(TOF_HFOV / 2)):.2f} m at 1 m, {2 * 3.0 * np.tan(np.radians(TOF_HFOV / 2)):.2f} m at 3 m')

fig, ax = plt.subplots(figsize=(7, 3.6))
a = np.radians(TOF_HFOV / 2)
ax.fill([0, TOF_RANGE, TOF_RANGE, 0], [0, TOF_RANGE * np.tan(a), -TOF_RANGE * np.tan(a), 0], alpha=0.3, label='ToF pair, 45 deg, to 4 m')
ax.axhspan(-PATH_W / 2, PATH_W / 2, color='tab:green', alpha=0.15, label='walking path, 0.6 m')
ax.fill([0, full_path, full_path, 0], [0, PATH_W / 2, -PATH_W / 2, 0], color='tab:red', alpha=0.4, label=f'path blind zone, to {full_path:.2f} m')
ax.annotate('', xy=(6.0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='k', lw=1.5))
ax.text(4.15, 0.45, chr(10).join(['radar, forward:', 'people to 5.2 m (R7)', 'vehicles to 23 m (R8)']), fontsize=8)
ax.axvline(3, color='gray', ls='--', lw=1, label='G1: 3 m')
ax.plot(0, 0, 'ks', label='chest unit')
ax.set_xlim(-0.2, 6)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.set_xlabel('distance ahead (m)')
ax.set_ylabel('lateral offset (m)')
ax.set_title('Top view: ToF coverage of the walking path')
ax.legend(fontsize=7, loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / 'tool2_topview.png', dpi=200)
plt.show()

## Tool 3: trade studies

Study A, far-field moving-object sensing (R7, R8). Study B, near-field obstacle and drop sensing (R1, R9). Weights were set from the requirements before scoring; reasons are in the journal, Part 3.

In [ ]:
def score_value(x, better, threshold, goal):
    """0 at the threshold, 1 at the goal, linear in between, clipped to the range 0 to 1."""
    if better == 'higher':
        s = (x - threshold) / (goal - threshold)
    else:
        s = (threshold - x) / (threshold - goal)
    return float(np.clip(s, 0.0, 1.0))


def meets(x, better, threshold):
    """True when x is on the right side of a must-meet threshold."""
    return x >= threshold if better == 'higher' else x <= threshold


def trade_study(candidates, criteria, n_trials=5000, spread=0.25, seed=0, title='Trade study', fname=None):
    """Must-meet screen, weighted scores, and a check of how much the winner depends on the weights."""
    w = np.array([c['weight'] for c in criteria], dtype=float)
    if not np.isclose(w.sum(), 1.0):
        raise ValueError(f'The weights add up to {w.sum():.3f}, not 1')
    names, rows = [], []
    for name, attrs in candidates.items():
        failed = [c['attr'] for c in criteria
                  if c['must'] and not meets(attrs[c['attr']], c['better'], c['threshold'])]
        if failed:
            print(f'Screened out: {name}, fails {", ".join(failed)}')
            continue
        names.append(name)
        rows.append([score_value(attrs[c['attr']], c['better'], c['threshold'], c['goal']) for c in criteria])
    if not names:
        print('No candidate meets every must-meet threshold.')
        return None
    S = np.array(rows)
    total = S @ w
    rng = np.random.default_rng(seed)
    wins = np.zeros(len(names))
    for _ in range(n_trials):
        wt = w * np.exp(spread * rng.standard_normal(w.size))
        wins[np.argmax(S @ (wt / wt.sum()))] += 1
    order = np.argsort(-total)
    width = max(len(n) for n in names)
    print()
    print('Candidate'.ljust(width) + '   score   wins')
    for i in order:
        print(f'{names[i]:{width}s}   {total[i]:5.3f}  {100 * wins[i] / n_trials:4.0f}%')
    share = wins[order[0]] / n_trials
    verdict = 'a clear choice' if share >= 0.7 else 'not a clear choice: decide with a test or a new criterion'
    print(f'Winner: {names[order[0]]}, in {100 * share:.0f}% of {n_trials} weight perturbations: {verdict}')

    fig, ax = plt.subplots(figsize=(9, 0.6 * len(names) + 1.6))
    left = np.zeros(len(names))
    for j, c in enumerate(criteria):
        if c['weight'] > 0:
            part = S[order, j] * w[j]
            ax.barh(np.arange(len(names)), part, left=left, label=c['attr'])
            left += part
    ax.set_yticks(np.arange(len(names)))
    ax.set_yticklabels([names[i] for i in order])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('weighted score (1 would be ideal on every criterion)')
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    fig.tight_layout()
    if fname:
        fig.savefig(FIGURES / fname, dpi=200)
    plt.show()
    return {names[i]: {'score': float(total[i]), 'win_share': float(wins[i] / n_trials)} for i in order}

In [ ]:
# Study A. Range and power: TI IWR6843AOP datasheet SWRS237C (1.19 W at 24 percent duty);
# radar range on a car at 23 m is assumption A10. Other ranges: ST DS13754 and the worked example.
far_candidates = {
    'mmWave radar, TI IWR6843AOP': {'range_m': 25, 'dark': 1, 'velocity': 1.0, 'rain': 0.9, 'wear': 0.7, 'power_w': 1.19},
    '2D lidar, RPLIDAR C1':        {'range_m': 6.0, 'dark': 1, 'velocity': 0.3, 'rain': 0.4, 'wear': 0.2, 'power_w': 1.0},
    'ToF array, VL53L5CX':         {'range_m': 4.0, 'dark': 1, 'velocity': 0.2, 'rain': 0.5, 'wear': 1.0, 'power_w': 0.3},
    'Depth camera, OAK-D Lite':    {'range_m': 3.0, 'dark': 0, 'velocity': 0.5, 'rain': 0.5, 'wear': 0.5, 'power_w': 3.0},
    'Ultrasonic, A02YYUW':         {'range_m': 4.5, 'dark': 1, 'velocity': 0.1, 'rain': 0.9, 'wear': 0.8, 'power_w': 0.04},
}
far_criteria = [
    {'attr': 'range_m',  'weight': 0.30, 'better': 'higher', 'threshold': vehicle, 'goal': 40, 'must': True},
    {'attr': 'dark',     'weight': 0.00, 'better': 'higher', 'threshold': 0.5, 'goal': 1, 'must': True},
    {'attr': 'velocity', 'weight': 0.25, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'rain',     'weight': 0.15, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'wear',     'weight': 0.10, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'power_w',  'weight': 0.20, 'better': 'lower',  'threshold': 2.0, 'goal': 0.2, 'must': True},
]
study_a = trade_study(far_candidates, far_criteria, title='Study A: far-field moving-object sensing (R7, R8)', fname='tool3_studyA.png')

In [ ]:
# Study B. ToF: ST DS13754 (4 m, 8 x 8 zones, 15 Hz); ToF power is an estimate. Other ranges: worked example.
near_candidates = {
    'ToF pair, 2 x VL53L5CX':        {'range_m': 4.0, 'dark': 1, 'vfov_cov': 1.0, 'thin': 0.3, 'drop': 0.6, 'wear': 1.0, 'power_w': 0.6},
    'Depth camera, OAK-D Lite':      {'range_m': 3.0, 'dark': 0, 'vfov_cov': 0.8, 'thin': 0.9, 'drop': 0.8, 'wear': 0.4, 'power_w': 3.0},
    '2D lidar, RPLIDAR C1':          {'range_m': 6.0, 'dark': 1, 'vfov_cov': 0.1, 'thin': 0.6, 'drop': 0.1, 'wear': 0.2, 'power_w': 1.0},
    'Ultrasonic array, 3 x A02YYUW': {'range_m': 2.0, 'dark': 1, 'vfov_cov': 0.6, 'thin': 0.4, 'drop': 0.2, 'wear': 0.8, 'power_w': 0.12},
    'ToF trio, 3 x VL53L5CX':        {'range_m': 4.0, 'dark': 1, 'vfov_cov': 1.0, 'thin': 0.35, 'drop': 0.7, 'wear': 0.9, 'power_w': 0.9},
}
near_criteria = [
    {'attr': 'range_m',  'weight': 0.15, 'better': 'higher', 'threshold': 3.0, 'goal': 6, 'must': True},
    {'attr': 'dark',     'weight': 0.00, 'better': 'higher', 'threshold': 0.5, 'goal': 1, 'must': True},
    {'attr': 'vfov_cov', 'weight': 0.25, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'thin',     'weight': 0.15, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'drop',     'weight': 0.15, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'wear',     'weight': 0.10, 'better': 'higher', 'threshold': 0, 'goal': 1, 'must': False},
    {'attr': 'power_w',  'weight': 0.20, 'better': 'lower',  'threshold': 1.5, 'goal': 0.1, 'must': True},
]
study_b = trade_study(near_candidates, near_criteria, title='Study B: near-field obstacle and drop sensing (R1, R9)', fname='tool3_studyB.png')

# Part 4: Localization, Mapping & Navigation

## Tool 4 and Tool 5: drift, and the pose entering the building

Outdoors the phone GNSS gives 3 m (1σ, assumption A8) at 1 Hz. Indoors the foot IMU with zero-velocity updates dead-reckons, with a 3 percent step-length error (A9). R10 asks for the walking distance at which the 2D 95 percent error passes 2 m.

In [ ]:
def gyro_noise(path='~/EECE5554/Lab3/analysis/noise_parameters.json', axis='z'):
    """N, B, and K of one gyro axis in SI units: Lab 3 results if they exist, else the VN-100 datasheet."""
    p = Path(path).expanduser()
    if p.exists():
        a = json.loads(p.read_text())[axis]
        return {'N': a['N'], 'B': a['B'], 'K': a.get('K') or 0.0, 'source': f'Lab 3, {p}'}
    return {'N': np.radians(0.0035), 'B': np.radians(10.0) / 3600.0, 'K': 0.0, 'source': 'VN-100 datasheet'}


def dead_reckoning_drift(t, v, N, B, K=0.0, sigma_y0=0.0, sigma_psi0=0.0, q_lat=0.0, scale_err=0.0):
    """1-sigma error growth for straight walking with an integrated gyro heading."""
    t = np.asarray(t, dtype=float)
    terms = {'start position': np.full_like(t, sigma_y0),
             'start heading': v * sigma_psi0 * t,
             'gyro bias B': v * B * t ** 2 / 2.0,
             'angle random walk N': v * N * np.sqrt(t ** 3 / 3.0),
             'rate random walk K': v * K * np.sqrt(t ** 5 / 20.0),
             'lateral slip': np.sqrt(q_lat * t)}
    cross = np.sqrt(sum(x ** 2 for x in terms.values()))
    along = np.sqrt(sigma_y0 ** 2 + (scale_err * v * t) ** 2)
    return terms, cross, along


def kalman_update(P, H, r):
    """Joseph-form covariance update for one scalar measurement with variance r."""
    S = (H @ P @ H.T).item() + r
    G = P @ H.T / S
    A = np.eye(P.shape[0]) - G @ H
    return A @ P @ A.T + r * (G @ G.T)


def fix_interval_analysis(v, N, B, K=0.0, q_lat=0.0, fix_sigma=0.02, fix_interval=1.0,
                          sigma_y0=0.02, sigma_psi0=np.radians(2.0), duration=300.0, dt=0.05, outages=()):
    """Covariance of a three-state Kalman filter: cross-track error, heading error, and gyro bias."""
    F = np.array([[1.0, v * dt, v * dt ** 2 / 2.0], [0.0, 1.0, dt], [0.0, 0.0, 1.0]])
    qn, qk = N ** 2, K ** 2
    Q = np.array([
        [q_lat * dt + qn * v ** 2 * dt ** 3 / 3.0 + qk * v ** 2 * dt ** 5 / 20.0,
         qn * v * dt ** 2 / 2.0 + qk * v * dt ** 4 / 8.0, qk * v * dt ** 3 / 6.0],
        [0.0, qn * dt + qk * dt ** 3 / 3.0, qk * dt ** 2 / 2.0],
        [0.0, 0.0, qk * dt]])
    Q = Q + np.triu(Q, 1).T
    P = np.diag([sigma_y0 ** 2, sigma_psi0 ** 2, B ** 2])
    H_pos = np.array([[1.0, 0.0, 0.0]])
    n = int(round(duration / dt))
    every_fix = max(1, int(round(fix_interval / dt)))
    t = np.arange(n + 1) * dt
    sig = np.empty((n + 1, 3))
    sig[0] = np.sqrt(np.diag(P))
    for k in range(1, n + 1):
        P = F @ P @ F.T + Q
        lost = any(start <= t[k] < end for start, end in outages)
        if k % every_fix == 0 and not lost:
            P = kalman_update(P, H_pos, fix_sigma ** 2)
        sig[k] = np.sqrt(np.diag(P))
    return t, sig[:, 0], sig[:, 1], sig[:, 2]

In [ ]:
gyro = gyro_noise()
N, B, K = gyro['N'], gyro['B'], gyro['K']
print(f'Gyro ({gyro["source"]}): N = {np.degrees(N) * 60:.3f} deg/sqrt(h), B = {np.degrees(B) * 3600:.1f} deg/h')

Q_LAT = 0.01 ** 2   # m^2/s, lateral noise of foot-IMU dead reckoning (A9)
STEP_ERR = 0.03     # step-length error, fraction of distance (A9)

# Tool 5: walking outdoors on 1 Hz phone GNSS at 3 m (A8), then entering the building at 200 s.
t, sy, spsi, sb = fix_interval_analysis(v=V_USER, N=N, B=B, K=K, q_lat=Q_LAT, fix_sigma=3.0, fix_interval=1.0,
                                        sigma_y0=3.0, sigma_psi0=np.radians(10), duration=300, outages=[(200, 1e9)])
i0 = np.searchsorted(t, 200) - 1
PSI_ENTER = spsi[i0]
print(f'Entering the building: position {sy[i0]:.2f} m, heading {np.degrees(PSI_ENTER):.2f} deg, '
      f'gyro bias {np.degrees(sb[i0]) * 3600:.1f} deg/h, all 1 sigma')

# Tool 4: indoors from the last structure fix, starting with that heading.
tt = np.linspace(0, 120, 1201)
terms, cross, along = dead_reckoning_drift(tt, V_USER, N, B, K, 0.10, PSI_ENTER, Q_LAT, STEP_ERR)
r95 = K95['2D'] * np.sqrt((cross ** 2 + along ** 2) / 2)
t_2m = tt[np.nonzero(r95 > 2.0)[0][0]]
j = np.searchsorted(tt, 60 / V_USER)
print(f'2D 95 percent error passes 2 m after {t_2m:.1f} s, {V_USER * t_2m:.1f} m of walking')
print(f'A 60 m indoor leg ends at {r95[j]:.2f} m (95 percent): R10 sets a structure fix at least every 35 m')

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, x in terms.items():
    if np.any(x > 0):
        ax.loglog(tt[1:], x[1:], lw=1, label=name)
ax.loglog(tt[1:], along[1:], lw=1, ls='--', label='along-track, 3 % step length')
ax.loglog(tt[1:], r95[1:], 'k', lw=2, label='2D, 95 percent')
ax.axhline(2.0, color='tab:red', ls='--', label='R10: 2 m, 95 percent')
ax.axvline(60 / V_USER, color='gray', ls=':', label='60 m indoor leg')
ax.set_xlabel('time since the last structure fix (s)')
ax.set_ylabel('position error (m)')
ax.set_title('Indoor dead reckoning: foot IMU with zero-velocity updates')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / 'tool4_indoor_drift.png', dpi=200)
plt.show()

## Tool 6: error budget for door arrival (R2)

R2: within 1 m of the door, 95 percent, horizontal radius, so k = 2.45. The final approach is relative to the door that the camera reads.

In [ ]:
def error_budget(contributors, requirement_95, dims='1D', title='Error budget', fname=None):
    """Root sum square of independent 1-sigma contributors, compared with a 95 percent requirement."""
    k = K95[dims]
    names = [c[0] for c in contributors]
    sig = np.array([c[1] for c in contributors], dtype=float)
    total = float(np.sqrt(np.sum(sig ** 2)))
    share = sig ** 2 / total ** 2
    width = max(len(n) for n in names)
    print('Contributor'.ljust(width) + '   1 sigma (m)  share  source')
    for (name, s, src), f in zip(contributors, share):
        print(f'{name:{width}s}   {s:6.3f}      {100 * f:4.0f}%  {src}')
    bound = k * total
    verdict = 'meets' if bound <= requirement_95 else 'FAILS'
    print(f'Total {total:.3f} m (1 sigma), {bound:.3f} m at 95 percent, against {requirement_95:.3f} m required: '
          f'{verdict}, margin {requirement_95 - bound:+.3f} m')
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(names) + 1.2))
    order = np.argsort(share)
    ax.barh([names[i] for i in order], 100 * share[order])
    ax.set_xlabel('share of variance (%)')
    ax.set_title(f'{title}: {bound:.2f} m at 95 percent, {requirement_95:.2f} m required')
    ax.grid(alpha=0.3, axis='x')
    fig.tight_layout()
    if fname:
        fig.savefig(FIGURES / fname, dpi=200)
    plt.show()
    return total, bound


def allowed_sigma(requirement_95, other_sigmas, dims='1D'):
    """Largest 1-sigma value one contributor may have once all the others are fixed."""
    room = (requirement_95 / K95[dims]) ** 2 - float(np.sum(np.asarray(other_sigmas, dtype=float) ** 2))
    return float(np.sqrt(room)) if room > 0 else 0.0

In [ ]:
_, c3, a3 = dead_reckoning_drift(np.array([0, 3 / V_USER]), V_USER, N, B, K, 0, PSI_ENTER, Q_LAT, STEP_ERR)
PDR_LAST_3M = float(np.hypot(c3[-1], a3[-1]))
door = [('user stopping variability', 0.222 * (1.4 / 1.22) ** 2, 'accuracy: E8 SD 0.222 m x (1.4/1.22)^2'),
        ('door position from sensing', 0.10, 'accuracy: assumption, test T2'),
        ('dead reckoning over the last 3 m', PDR_LAST_3M, 'accuracy: Tool 4'),
        ('cue latency jitter, 50 ms x 1.4 m/s', 0.07, 'accuracy: assumption, test T4'),
        ('chest-to-body-centre offset', 0.05, 'accuracy: assumption, tape measure')]
error_budget(door, 1.0, dims='2D', title='R2 door arrival', fname='tool6_door_budget.png')
print(f'The stopping spread may grow to {allowed_sigma(1.0, [s for _, s, _ in door[1:]], "2D"):.3f} m (1 sigma) before R2 fails')

# Part 5: System Integration

## Tool 7: data rates over Bluetooth Low Energy

In [ ]:
def data_budget(topics, link_mbps=None):
    """Bandwidth of each topic and the total over one link. topics: (name, bytes per message, rate in Hz)."""
    width = max(len(x[0]) for x in topics)
    total = 0.0
    for name, size, rate in topics:
        kbps = 8 * size * rate / 1e3
        total += kbps
        print(f'{name:{width}s}  {kbps:7.1f} kbit/s')
    note = f', {100 * total / (1e3 * link_mbps):.0f} percent of {link_mbps} Mbit/s' if link_mbps else ''
    print(f'Total {total / 1e3:.3f} Mbit/s{note}')
    return total


ble = [('radar tracks, 20 x 16 B', 320, 10),
       ('ToF zones, 2 x 64 x 3 B', 384, 15),
       ('sign image crop, near doors', 5000, 1),
       ('foot IMU step vectors', 40, 2),
       ('barometer', 16, 10),
       ('heartbeat and mode', 8, 10),
       ('cues to two wristbands', 16, 10)]
data_budget(ble, link_mbps=0.5)   # BLE application throughput of 0.5 Mbit/s is an assumption to measure

## Tool 8: power and runtime of the chest unit

Three Samsung INR18650-35E cells in parallel: 3.6 V, 3.35 Ah minimum and 8 A continuous each (Samsung SDI specification). R4: 12 h on an aged pack at 2.8 °C.

In [ ]:
def power_budget(loads, battery, converter_eff=0.90, required_min=None, reserve=0.15, design_case='aged pack, cold'):
    """Average and peak power, and runtime on a new pack, an aged pack, and an aged pack in the cold."""
    width = max(len(x[0]) for x in loads)
    print('Load'.ljust(width) + '   avg W  peak W')
    avg = peak = 0.0
    for name, a, p, on_converter in loads:
        f = 1.0 / converter_eff if on_converter else 1.0
        avg += a * f
        peak += p * f
        print(f'{name:{width}s}  {a * f:6.2f}  {p * f:6.2f}')
    wh = battery['volts'] * battery['amp_hours']
    usable = wh * battery['usable']
    cases = {'new pack': usable,
             'aged pack': usable * battery['aged'],
             'aged pack, cold': usable * battery['aged'] * battery['cold']}
    print(f'Total {avg:.2f} W average and {peak:.2f} W peak, from a {wh:.1f} Wh pack')
    minutes = {}
    for case, e in cases.items():
        minutes[case] = 60.0 * e * (1.0 - reserve) / avg
        print(f'   {case:16s} {e:6.1f} Wh usable  {minutes[case] / 60:5.1f} h before the reserve')
    amps = peak / battery['volts']
    c_rate = amps / battery['amp_hours']
    limit = battery['max_c_rate']
    print(f'Peak current {amps:.2f} A, which is {c_rate:.2f}C: {"within" if c_rate <= limit else "OVER"} the {limit:.2f}C limit')
    if required_min is not None:
        got = minutes[design_case]
        print(f'Requirement: {required_min / 60:.0f} h on the {design_case}: {"meets" if got >= required_min else "FAILS"} with {got / 60:.1f} h')
    return minutes


pack = {'volts': 3.6, 'amp_hours': 3 * 3.35, 'usable': 0.90, 'aged': 0.80, 'cold': 0.85, 'max_c_rate': 3 * 8.0 / (3 * 3.35)}
for radar_w, label in ((1.19, 'radar at the datasheet 24 percent duty'), (0.60, 'radar duty-cycled, estimate')):
    print(label)
    loads = [('mmWave radar', radar_w, 1.75, True),
             ('ToF sensors, 2', 0.60, 0.60, True),
             ('sign camera, 10 percent duty', 0.03, 0.30, True),
             ('safety MCU and BLE', 0.03, 0.06, True),
             ('barometer', 0.001, 0.001, True)]
    power_budget(loads, pack, required_min=12 * 60)
    print()

print('Wristband, each: LiPo 3.7 V, 150 mAh (estimate)')
band = [('MCU and BLE', 0.01, 0.02, True), ('LRA motor, 5 percent duty', 0.005, 0.10, True)]
power_budget(band, {'volts': 3.7, 'amp_hours': 0.15, 'usable': 0.90, 'aged': 0.80, 'cold': 0.85, 'max_c_rate': 1.0}, required_min=12 * 60)
print()
print('Foot IMU pod: LiPo 3.7 V, 250 mAh (estimate); product IMU, not the VN-100')
foot = [('MEMS IMU and ZUPT on the MCU', 0.02, 0.02, True), ('BLE', 0.01, 0.02, True)]
power_budget(foot, {'volts': 3.7, 'amp_hours': 0.25, 'usable': 0.90, 'aged': 0.80, 'cold': 0.85, 'max_c_rate': 1.0}, required_min=12 * 60)

## Tool 9: prototype parts list

In [ ]:
def parts_list(items):
    """Out-of-pocket cost, added mass, and added power of a prototype."""
    wp = max(len(i['part']) for i in items)
    wm = max(len(i['model']) for i in items)
    print('Part'.ljust(wp) + '  ' + 'Model'.ljust(wm) + '  qty    unit $  owned  mass kg  power W  source')
    cost = mass = power = 0.0
    for i in items:
        u = i['unit_usd']
        unit = '      n/a' if u is None else f'{u:9.2f}'
        if not i['owned'] and u is not None:
            cost += i['qty'] * u
        mass += i['qty'] * i['mass_kg']
        power += i['qty'] * i['power_w']
        print(f"{i['part']:{wp}s}  {i['model']:{wm}s}  {i['qty']:3d}  {unit}  {'yes' if i['owned'] else 'no':>5s}  "
              f"{i['mass_kg']:7.3f}  {i['power_w']:7.2f}  {i['source']}")
    print(f'Out of pocket ${cost:.2f}, added mass {mass:.3f} kg, added power {power:.2f} W')
    return cost, mass, power


prototype = [
    {'part': 'Foot IMU', 'model': 'VectorNav VN-100', 'qty': 1, 'unit_usd': None, 'mass_kg': 0.015, 'power_w': 0.20, 'source': 'course, Lab 3', 'owned': True},
    {'part': 'Phone', 'model': 'own phone', 'qty': 1, 'unit_usd': None, 'mass_kg': 0.0, 'power_w': 0.0, 'source': 'own', 'owned': True},
    {'part': 'Radar', 'model': 'TI IWR6843AOPEVM', 'qty': 1, 'unit_usd': 178.80, 'mass_kg': 0.020, 'power_w': 1.19, 'source': 'DigiKey; 1.19 W at the datasheet 24 percent duty; 0.60 W duty-cycled in the product (A11)', 'owned': False},
    {'part': 'ToF sensor', 'model': 'VL53L5CX breakout', 'qty': 2, 'unit_usd': 25.00, 'mass_kg': 0.003, 'power_w': 0.30, 'source': 'estimate', 'owned': False},
    {'part': 'MCU with BLE', 'model': 'nRF52840 board', 'qty': 1, 'unit_usd': 25.00, 'mass_kg': 0.005, 'power_w': 0.03, 'source': 'estimate', 'owned': False},
    {'part': 'Sign camera', 'model': 'small camera module with IR LED', 'qty': 1, 'unit_usd': 15.00, 'mass_kg': 0.010, 'power_w': 0.03, 'source': 'estimate; 0.30 W at 10 percent duty', 'owned': False},
    {'part': 'Barometer', 'model': 'barometer breakout', 'qty': 1, 'unit_usd': 10.00, 'mass_kg': 0.001, 'power_w': 0.001, 'source': 'estimate', 'owned': False},
    {'part': 'Battery cell', 'model': 'Samsung INR18650-35E', 'qty': 3, 'unit_usd': 5.00, 'mass_kg': 0.050, 'power_w': 0.0, 'source': 'Samsung SDI, 50 g max; price estimated', 'owned': False},
    {'part': 'Wristband motor', 'model': 'LRA with driver', 'qty': 2, 'unit_usd': 10.00, 'mass_kg': 0.010, 'power_w': 0.10, 'source': 'estimate', 'owned': False},
    {'part': 'Straps and mounts', 'model': 'strap and printed mounts', 'qty': 1, 'unit_usd': 15.00, 'mass_kg': 0.045, 'power_w': 0.0, 'source': 'estimate', 'owned': False},
]
parts_list(prototype)

worn = [('3 x Li-ion cell, INR18650-35E', 150, 'datasheet, 50 g max each'), ('radar module', 20, 'estimate'),
        ('2 x ToF breakout', 6, 'estimate'), ('sign camera with IR LED', 10, 'estimate'),
        ('safety MCU with BLE, barometer', 6, 'estimate'), ('chest enclosure and strap', 45, 'estimate'),
        ('2 x wristband', 40, 'estimate'), ('foot IMU pod', 15, 'estimate')]
total_g = sum(m for _, m, _ in worn)
print(f'Worn mass in the product: {total_g} g against 300 g, {300 - total_g} g to spare')

## Tool 10: traceability

In [ ]:
def traceability(requirements, design, verification):
    """Every requirement met by something and checked by something, and every part there for a reason."""
    ids = list(requirements)
    width = max(len(e) for e in list(design) + ['verification'])
    print(' ' * (width + 1) + ''.join(f'{r:>5s}' for r in ids))
    for element, served in design.items():
        print(f'{element:{width}s} ' + ''.join(f'{"x" if r in served else ".":>5s}' for r in ids))
    print(f'{"verification":{width}s} ' + ''.join(f'{"x" if r in verification else ".":>5s}' for r in ids))
    print()
    problems = []
    for r in ids:
        if not any(r in served for served in design.values()):
            problems.append(f'{r} has no design element')
        if r not in verification:
            problems.append(f'{r} has no verification')
    for element, served in design.items():
        if not served:
            problems.append(f'{element} serves no requirement')
        for r in served:
            if r not in requirements:
                problems.append(f'{element} lists {r}, which is not a requirement ID')
    for p in problems:
        print(p)
    print('Traceability: no problems found' if not problems else f'Traceability: {len(problems)} problems to fix')
    return problems


requirements = {
    'R1': 'Warn of obstacles, ground to 1.65 m, within 3 m at 1.4 m/s',
    'R2': 'Arrive within 1 m of the door, 95 percent, horizontal',
    'R3': 'All worn parts under 300 g',
    'R4': 'At least 12 h on an aged pack at 2.8 C',
    'R5': 'No phone handling at any point',
    'R6': 'Near-field latency at most 0.989 s',
    'R7': 'Warn of an oncoming person at 5.17 m or more',
    'R8': 'Warn of a vehicle at a driveway at 23.0 m or more',
    'R9': 'Warn of a drop of 150 mm or more at 3 m or more',
    'R10': 'A structure fix at least every 35 m indoors',
}
design = {
    'ToF pair': ['R1', 'R6', 'R9', 'R10'],
    'mmWave radar': ['R7', 'R8'],
    'sign camera': ['R2'],
    'foot IMU with ZUPT': ['R2', 'R10'],
    'phone GNSS': ['R2'],
    'safety MCU and band watchdogs': ['R1', 'R6'],
    'three-cell pack, radar duty cycle': ['R4'],
    'mass budget': ['R3'],
    'phone in pocket, wristband buttons': ['R5'],
}
verification = {
    'R1': 'T1', 'R2': 'T2', 'R3': 'inspection, weigh every part', 'R4': 'Tool 8, then T3', 'R5': 'T2',
    'R6': 'T4', 'R7': 'T5', 'R8': 'T6', 'R9': 'T7', 'R10': 'T8',
}
problems = traceability(requirements, design, verification)

# Part 6: Failure Analysis

## Tool 11: failure modes and effects

In [ ]:
def fmea(rows, top=10, fname=None):
    """Risk priority numbers before and after each action, ranked, with every severity 9 or 10 row flagged."""
    for r in rows:
        r['RPN'] = r['S'] * r['O'] * r['D']
        r['RPN2'] = r['S2'] * r['O2'] * r['D2']
    ranked = sorted(rows, key=lambda r: (-r['RPN'], -r['S']))
    width = max(len(r['mode']) for r in ranked)
    print('Failure mode'.ljust(width) + '    S  O  D  RPN  after')
    for r in ranked:
        flag = '   severity 9 or 10' if r['S'] >= 9 else ''
        print(f"{r['mode']:{width}s}  {r['S']:3d}{r['O']:3d}{r['D']:3d}  {r['RPN']:3d}  {r['RPN2']:5d}{flag}")
    print()
    print('Severity 9 or 10:')
    for r in ranked:
        if r['S'] >= 9:
            print(f"   {r['mode']}: detected by {r['detection']}; action: {r['action']}")
    shown = ranked[:top]
    y = np.arange(len(shown))
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(shown) + 1.2))
    ax.barh(y, [r['RPN'] for r in shown], color='lightgray', label='before')
    ax.barh(y, [r['RPN2'] for r in shown], height=0.5, color='tab:blue', label='after the action')
    ax.set_yticks(y)
    ax.set_yticklabels([r['mode'] for r in shown], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('risk priority number, S x O x D')
    ax.set_title(f'The top {len(shown)} failure modes')
    ax.legend()
    fig.tight_layout()
    if fname:
        fig.savefig(FIGURES / fname, dpi=200)
    plt.show()
    return ranked


failures = [
    {'mode': 'Drop of 13 to 150 mm missed', 'S': 9, 'O': 5, 'D': 8, 'S2': 9, 'O2': 4, 'D2': 6,
     'detection': 'foot IMU stumble: vertical acceleration spike with no step pattern', 'action': 'stop cue, safe fallback'},
    {'mode': 'Thin branch or pole at head height missed', 'S': 9, 'O': 5, 'D': 7, 'S2': 9, 'O2': 4, 'D2': 6,
     'detection': 'radar return at head height with no ToF return', 'action': 'slow-down cue'},
    {'mode': 'GNSS biased between buildings', 'S': 6, 'O': 7, 'D': 7, 'S2': 6, 'O2': 7, 'D2': 4,
     'detection': '99 percent innovation gate, 3 rejections in a row', 'action': 'reject, degraded position'},
    {'mode': 'Radar false alarms in crowds', 'S': 7, 'O': 6, 'D': 6, 'S2': 7, 'O2': 4, 'D2': 4,
     'detection': 'tracks confirmed over 3 frames', 'action': 'cue only closing tracks in the path cone'},
    {'mode': 'User misreads a vibration cue', 'S': 8, 'O': 4, 'D': 7, 'S2': 8, 'O2': 3, 'D2': 4,
     'detection': 'no button confirmation of a turn within 3 s', 'action': 'repeat the cue'},
    {'mode': 'Sun blinds the ToF sensors', 'S': 9, 'O': 4, 'D': 5, 'S2': 9, 'O2': 4, 'D2': 3,
     'detection': 'valid zones under 50 percent for 0.5 s', 'action': 'slow-down cue; radar still warns'},
    {'mode': 'Vehicle above 20 mph or radar range short', 'S': 10, 'O': 3, 'D': 6, 'S2': 10, 'O2': 2, 'D2': 5,
     'detection': 'radar closing speed over 8.94 m/s', 'action': 'stop cue at once; range test T6'},
    {'mode': 'Wrong door, number misread', 'S': 6, 'O': 4, 'D': 7, 'S2': 6, 'O2': 4, 'D2': 3,
     'detection': 'number read twice and position within 2 m', 'action': 'not-confirmed cue'},
    {'mode': 'Indoor step-length drift', 'S': 5, 'O': 6, 'D': 5, 'S2': 5, 'O2': 3, 'D2': 3,
     'detection': '95 percent bound over the threshold', 'action': 'structure fix every 35 m (R10)'},
    {'mode': 'Floor misidentified by the barometer', 'S': 5, 'O': 4, 'D': 5, 'S2': 5, 'O2': 2, 'D2': 3,
     'detection': 'floor sign read in the elevator lobby', 'action': 'correct the floor'},
    {'mode': 'Wristband motor fails', 'S': 9, 'O': 2, 'D': 5, 'S2': 9, 'O2': 2, 'D2': 2,
     'detection': 'motor current at start and every 60 s', 'action': 'other band plays guidance unavailable'},
    {'mode': 'Phone app crashes or BLE drops', 'S': 8, 'O': 4, 'D': 2, 'S2': 8, 'O2': 4, 'D2': 1,
     'detection': 'no heartbeat for 2 periods, seen by the bands and the chest MCU', 'action': 'chest MCU keeps STOP'},
    {'mode': 'Battery runs out early, aged and cold', 'S': 6, 'O': 3, 'D': 3, 'S2': 6, 'O2': 2, 'D2': 2,
     'detection': 'charge at the Tool 8 reserve', 'action': 'low-battery cue'},
]
ranked = fmea(failures, fname='tool11_fmea.png')